In [1]:
# Will output GPU info if Colab is connected correctly.
# Otherwise, outputs an error.
!nvidia-smi

Tue Nov 10 04:09:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   67C    P8             18W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
### MOUNT GOOGLE DRIVE SO WORK IS SAVED

from google.colab import drive
import os
from pathlib import Path

drive.mount('/content/drive')

DRIVE_DIR = Path('/content/drive/MyDrive/MelanomaClassfierProject')

DATA_DIR = DRIVE_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
FOLDS_DIR = DATA_DIR / 'folds'

IMAGE_DIRS = [
    RAW_DIR / 'HAM10000_images_part_1',
    RAW_DIR / 'HAM10000_images_part_2'
]

CHECKPOINT_DIR = DRIVE_DIR / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print("Drive setup complete!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive setup complete!


In [3]:
for folder_name in [
  "HAM10000_images_part_1",
  "HAM10000_images_part_2"
]:
  folder = DRIVE_DIR / "data/raw" / folder_name
  print(folder_name, len(list(folder.glob("*.jpg"))))

HAM10000_images_part_1 5000
HAM10000_images_part_2 5015


In [4]:
### PREPROCESSING SETUP

import sys
import importlib

# Add src directory on my Drive to Python import path.
sys.path.insert(0, str(DRIVE_DIR / 'src'))

import preprocessing as preproc
importlib.reload(preproc)

preproc.set_directories(base_directory=DRIVE_DIR)


Training and evaluation transforms created!
Training and evaluation transforms created!
Configuration saved!


In [ ]:
import time
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision.models as models

import numpy as np
from pathlib import Path
from sklearn.metrics import confusion_matrix, f1_score


### CONSTANTS
FOLD_NUM = 0  # Set to 0 so as not to overwrite current checkpoint files.
CLASSES_COUNT = 7  # nv, mel, bkl, bcc, akiec, vasc, df -- 7 total
FOLDS_COUNT = 1  # Eventually will be 5, but is set to 1 for the MVP.
EPOCHS_COUNT = 5
EFFICIENTNET_B0 = "efficientnet_b0"


### GLOBALS
best_recall = -1
best_macro_f1 = -1


def create_model() -> models.EfficientNet:
  backbone = preproc.CONFIG["backbone"]

  if backbone == EFFICIENTNET_B0:
    model = models.efficientnet_b0(
      weights=models.EfficientNet_B0_Weights.DEFAULT
    )
  # Add support for Efficientnet_b3 later.
  else:
    raise ValueError(f"Unsupported backbone: {backbone}")

  # Freeze early feature layers.
  for param in model.features.parameters():
    param.requires_grad = False

  # Leave later feature blocks unfrozen.
  for param in model.features[-2:].parameters():
    param.requires_grad = True

  # Make the final classification layer of the model match the number of classes
  # that we have.
  in_features = model.classifier[1].in_features
  model.classifier[1] = nn.Linear(in_features, CLASSES_COUNT)

  return model


def should_save_checkpoint(metrics):
  global best_recall, best_macro_f1

  recall = metrics["mean_high_risk_recall"]
  specificity = metrics["mean_high_risk_specificity"]
  macro_f1 = metrics["macro_f1"]

  if specificity < 0.80:
    return False

  if recall > best_recall or (
    np.isclose(recall, best_recall) and macro_f1 > best_macro_f1
  ):
    best_recall = recall
    best_macro_f1 = macro_f1
    return True

  return False


def evaluate_model(model, loader, criterion, device):
  # Switch model to evaluation mode.
  model.eval()

  total_loss = 0
  total_samples = 0

  all_labels = []
  all_predictions = []

  # Disable gradient calculations during evaluation.
  with torch.no_grad():
    for images, labels in loader:
      # Move batches to GPU.
      images = images.to(device)
      labels = labels.to(device)

      # Forward pass.
      outputs = model(images)

      # Calculate loss.
      loss = criterion(outputs, labels)

      # Get predicted classes.
      predictions = outputs.argmax(dim=1)

      total_loss += loss.item() * images.size(0)
      total_samples += images.size(0)

      # Collect predictions and actual labels.
      all_labels.extend(labels.cpu().numpy())
      all_predictions.extend(predictions.cpu().numpy())

  average_loss = total_loss / total_samples

  return (
      average_loss,
      np.array(all_labels),
      np.array(all_predictions)
  )


def calculate_metrics(y_true, y_pred):
  # Generate confusion matrix for all 7 classes.
  cm = confusion_matrix(
      y_true,
      y_pred,
      labels=list(range(7))
  )

  recalls = []
  specificities = []

  # Calculate recall and specificity for each class.
  for class_id in range(7):
    tp = cm[class_id, class_id]
    fn = cm[class_id, :].sum() - tp
    fp = cm[:, class_id].sum() - tp
    tn = cm.sum() - tp - fn - fp

    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    recalls.append(recall)
    specificities.append(specificity)

  # High-risk classes:
  # mel = 1, bcc = 3, akiec = 4
  high_risk_classes = [1, 3, 4]

  mean_high_risk_recall = np.mean([
    recalls[i] for i in high_risk_classes
  ])

  mean_high_risk_specificity = np.mean([
    specificities[i] for i in high_risk_classes
  ])

  # Calculate macro-F1 across all 7 classes.
  macro_f1 = f1_score(
    y_true,
    y_pred,
    labels=list(range(7)),
    average="macro",
    zero_division=0
  )

  return {
    "recalls": recalls,
    "specificities": specificities,
    "mean_high_risk_recall": mean_high_risk_recall,
    "mean_high_risk_specificity": mean_high_risk_specificity,
    "macro_f1": macro_f1
  }


# Train based on fold #<FOLD_NUM>.
train_df, val_df, test_df = preproc.init_fold(fold_number=FOLD_NUM)

sampler = preproc.do_oversampling(train_df)

train_loader, val_loader, test_loader = preproc.create_dataloaders(
  train_df = train_df,
  val_df = val_df,
  test_df = test_df,
  sampler = sampler
)

if torch.cuda.is_available():
  device = torch.device("cuda")
  print("Using device:", device)
  print("GPU:", torch.cuda.get_device_name(0))
  print("CUDA version:", torch.version.cuda)
  print("GPU memory:", round(
    torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
  ), "GB")
else:
  device = torch.device("cpu")
  print("Using device:", device)


## CREATE MODEL
model = create_model()


## INSTANTIATE MODEL
model.to(device)

print("Backbone:", preproc.CONFIG["backbone"])
print("Device:", next(model.parameters()).device)

optimizer = torch.optim.AdamW(
  filter(lambda p: p.requires_grad, model.parameters()),
  lr=1e-4,
  weight_decay=1e-4
)

criterion = nn.CrossEntropyLoss()


## CHECK FOR AN EXISTING CHECKPOINT

last_checkpoint_path = (
  CHECKPOINT_DIR / f"last_checkpoint_fold_{FOLD_NUM}.pth"
)

start_epoch = 1
best_recall = -1
best_macro_f1 = -1

if last_checkpoint_path.exists():
  print("Previous checkpoint found. Loading training progress...")

  checkpoint = torch.load(
      last_checkpoint_path,
      map_location=device,
      weights_only=False
  )

  model.load_state_dict(checkpoint["model_state_dict"])
  optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

  start_epoch = checkpoint["epoch"] + 1

  best_recall = checkpoint["best_recall"]
  best_macro_f1 = checkpoint["best_macro_f1"]

  print(f"Resuming from Epoch {start_epoch}")

else:
  print("No previous checkpoint found. Starting from Epoch 1.")


## TRAIN EPOCHS

for epoch in range(start_epoch, EPOCHS_COUNT + 1):
  model.train()

  total_loss = 0
  start_time = time.time()

  for batch_idx, (images, labels) in enumerate(train_loader):
    images = images.to(device)
    labels = labels.to(device)

    optimizer.zero_grad()

    # Forward pass.
    outputs = model(images)
    loss = criterion(outputs, labels)
    # Backpropagation.
    loss.backward()
    # Update trainable parameters.
    optimizer.step()

    total_loss += loss.item()

    if (batch_idx + 1) % 10 == 0:
      elapsed = time.time() - start_time

      print(
        f"Batch [{batch_idx + 1}/{len(train_loader)}] "
        f"Loss: {loss.item():.4f} "
        f"Elapsed: {elapsed:.1f}s",
        flush=True
      )

  average_train_loss = total_loss / (batch_idx + 1)


  # VALIDATE MODEL
  val_loss, y_true, y_pred = evaluate_model(
    model=model,
    loader=val_loader,
    criterion=criterion,
    device=device
  )

  # Calculate validation metrics.
  metrics = calculate_metrics(y_true, y_pred)

  print(f"Training Loss: {average_train_loss:.4f}")
  print(f"Validation Loss: {val_loss:.4f}")

  print(
    f"High-Risk Mean Recall: "
    f"{metrics['mean_high_risk_recall']:.4f}"
  )

  print(
    f"High-Risk Mean Specificity: "
    f"{metrics['mean_high_risk_specificity']:.4f}"
  )

  print(f"Macro F1: {metrics['macro_f1']:.4f}")

  if should_save_checkpoint(metrics):
    checkpoint = {
      "epoch": epoch,
      "model_state_dict": model.state_dict(),
      "optimizer_state_dict": optimizer.state_dict(),
      "val_loss": val_loss,
      "metrics": metrics,
      "config": preproc.CONFIG
    }

    checkpoint_path = (
      CHECKPOINT_DIR / f"best_model_fold_{FOLD_NUM}.pth"
    )

    torch.save(checkpoint, checkpoint_path)

    print(f"Best checkpoint saved to: {checkpoint_path}")


  # ALWAYS SAVE THE LATEST COMPLETED EPOCH
  last_checkpoint_path = (
      CHECKPOINT_DIR / f"last_checkpoint_fold_{FOLD_NUM}.pth"
  )

  torch.save({
    "epoch": epoch,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "val_loss": val_loss,
    "metrics": metrics,
    "best_recall": best_recall,
    "best_macro_f1": best_macro_f1,
    "config": preproc.CONFIG
  }, last_checkpoint_path)

  print(f"Latest checkpoint saved after Epoch {epoch}.")

Fold 5
Training: 6868
Validation: 1145
Testing: 2002
No lesion leakage detected!
Class-aware oversampling disabled.
Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])
Using device: cuda
GPU: Tesla T4
CUDA version: 13.0
GPU memory: 14.56 GB
Backbone: efficientnet_b0
Device: cuda:0
No previous checkpoint found. Starting from Epoch 1.
Batch [10/215] Loss: 1.7566 Elapsed: 4.9s
Batch [20/215] Loss: 1.4333 Elapsed: 9.1s
Batch [30/215] Loss: 1.4159 Elapsed: 13.4s
Batch [40/215] Loss: 1.2520 Elapsed: 18.3s
Batch [50/215] Loss: 1.0208 Elapsed: 22.4s
Batch [60/215] Loss: 1.0290 Elapsed: 26.5s
Batch [70/215] Loss: 0.9980 Elapsed: 31.5s
Batch [80/215] Loss: 1.1753 Elapsed: 35.8s
Batch [90/215] Loss: 0.7920 Elapsed: 39.9s
Batch [100/215] Loss: 0.8643 Elapsed: 44.5s
Batch [110/215] Loss: 0.9631 Elapsed: 48.9s
Batch [120/215] Loss: 1.0594 Elapsed: 52.9s
Batch [130/215] Loss: 0.9312 Elapsed: 57.3s
Batch [140/215] Loss: 0.8868 Elapsed: 62.0s
Batch [150/215] Loss: 0.719